# MT5 Spread Logger — All Symbols

این نوت‌بوک:
- به MetaTrader 5 وصل می‌شود
- لیست **همه‌ی symbols** را از بروکر می‌گیرد
- bid/ask و spread هر کدام را به صورت live ثبت می‌کند
- هم spread خام (price)، هم تعداد points بروکر، و هم تخمین pips را ذخیره می‌کند

قبل از اجرا:

```bash
pip install MetaTrader5 pandas
```

همچنین:
- MT5 باید باز باشد
- داخل MT5 لاگین کرده باشید
- Algo Trading فعال باشد


In [1]:

import MetaTrader5 as mt5
import pandas as pd
from datetime import datetime
import time


In [2]:

# اتصال به MT5
connected = mt5.initialize()

if not connected:
    print("MT5 initialize failed")
    print(mt5.last_error())
else:
    print("Connected to MT5")


Connected to MT5


In [3]:
# گرفتن لیست همه‌ی symbols از بروکر
all_symbols = mt5.symbols_get()

if all_symbols is None:
    print("symbols_get failed:", mt5.last_error())
    all_symbols = []

print(f"Total symbols available from broker: {len(all_symbols)}")


Total symbols available from broker: 1102


In [4]:
# فعال کردن همه‌ی symbols داخل Market Watch تا tick قابل خواندن باشد
activated = 0
for s in all_symbols:
    if not s.visible:
        if mt5.symbol_select(s.name, True):
            activated += 1

print(f"Activated {activated} symbols in Market Watch")


Activated 1096 symbols in Market Watch


In [ ]:
# یک snapshot لحظه‌ای از spread همه‌ی symbols
snapshot = []
now = datetime.now()

for s0 in all_symbols:
    # refresh: s0.spread از زمان symbols_get() است و آپدیت نمی‌شود
    s = mt5.symbol_info(s0.name)
    if s is None:
        continue

    tick = mt5.symbol_info_tick(s.name)
    if tick is None or tick.bid == 0 or tick.ask == 0:
        continue

    spread_price = tick.ask - tick.bid
    pip_size = s.point * 10 if s.point else None
    spread_pips = (spread_price / pip_size) if pip_size else None

    snapshot.append({
        "time": now,
        "symbol": s.name,
        "bid": tick.bid,
        "ask": tick.ask,
        "spread_price": spread_price,
        "spread_points": s.spread,   # refreshed broker int spread
        "spread_pips": round(spread_pips, 2) if spread_pips is not None else None,
        "digits": s.digits,
        "point": s.point,
        "trade_mode": s.trade_mode,  # 0=disabled, 4=full
    })

snap_df = pd.DataFrame(snapshot)
print(f"Got tick for {len(snap_df)} / {len(all_symbols)} symbols")
snap_df


## گرفتن اسپرد همه‌ی symbols هر چند ثانیه

این بخش هر `INTERVAL` ثانیه یک sample از همه‌ی symbols ذخیره می‌کند.
برای stop کردن:
- کرنل را متوقف کنید


In [ ]:
records = []

N_SAMPLES = 20
INTERVAL = 5  # seconds between samples

for i in range(N_SAMPLES):
    ts = datetime.now()
    got = 0

    for s0 in all_symbols:
        s = mt5.symbol_info(s0.name)   # refresh so .spread is current
        if s is None:
            continue

        tick = mt5.symbol_info_tick(s.name)
        if tick is None or tick.bid == 0 or tick.ask == 0:
            continue

        spread_price = tick.ask - tick.bid
        pip_size = s.point * 10 if s.point else None
        spread_pips = (spread_price / pip_size) if pip_size else None

        records.append({
            "time": ts,
            "symbol": s.name,
            "bid": tick.bid,
            "ask": tick.ask,
            "spread_price": spread_price,
            "spread_points": s.spread,
            "spread_pips": round(spread_pips, 4) if spread_pips is not None else None,
        })
        got += 1

    print(f"{ts}  sample {i+1}/{N_SAMPLES}  -> {got}/{len(all_symbols)} symbols")

    if i < N_SAMPLES - 1:
        time.sleep(INTERVAL)

df = pd.DataFrame(records)
print(f"Total rows: {len(df)}")
df


In [7]:
# ذخیره خام تمام sample ها
df.to_csv("spread_log_all_symbols.csv", index=False)
print("saved -> spread_log_all_symbols.csv")

# خلاصه آماری per-symbol (median / mean / max)
if not df.empty:
    summary = (
        df.groupby("symbol")
          .agg(
              samples=("spread_price", "count"),
              median_pips=("spread_pips", "median"),
              mean_pips=("spread_pips", "mean"),
              max_pips=("spread_pips", "max"),
              median_points=("spread_points", "median"),
              max_points=("spread_points", "max"),
          )
          .sort_values("median_pips")
    )
    summary.to_csv("spread_summary_all_symbols.csv")
    print("saved -> spread_summary_all_symbols.csv")
    summary


NameError: name 'df' is not defined


## نکات مهم

اگر الان market بسته باشد:
- spread معمولاً خیلی زیاد می‌شود
- عدد واقعی market نیست

بهترین زمان اندازه‌گیری:
- London Session
- New York Session
- زمان overlap لندن و نیویورک

برای backtest واقعی:
- median spread
- average spread
- max spread during volatility

را ذخیره کنید.


In [ ]:

# پایان اتصال
mt5.shutdown()
print("MT5 disconnected")
